<a href="https://colab.research.google.com/github/ryueuitae/project/blob/main/sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install implicit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 81.1 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import implicit
from scipy.sparse import csr_matrix

rate_df = pd.read_csv('/content/rate_data.csv', encoding='utf-8-sig')
item_df = pd.read_csv('/content/item_data.csv', encoding='utf-8-sig')

print("로드 완료!")
print(rate_df.shape)

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


로드 완료!
(174912, 4)


In [6]:
user_ids = rate_df['user'].astype('category').cat.codes
item_ids = rate_df['item'].astype('category').cat.codes

sparse_matrix = csr_matrix(
    (rate_df['rate'], (user_ids, item_ids))
)

model = implicit.als.AlternatingLeastSquares(
    factors=50,
    iterations=30,
    use_gpu=False
)
model.fit(sparse_matrix)
print("학습 완료!")

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

학습 완료!


In [7]:
user_cat = rate_df['user'].astype('category').cat
item_cat = rate_df['item'].astype('category').cat
user_map = dict(enumerate(user_cat.categories))
item_map = dict(enumerate(item_cat.categories))

def recommend_items(user_id, n=10):
    user_idx = list(user_cat.categories).index(user_id)
    ids, scores = model.recommend(
        user_idx, sparse_matrix[user_idx], N=n, filter_already_liked_items=True
    )
    top_item_ids = [item_map[i] for i in ids]
    result = item_df[item_df['item'].isin(top_item_ids)][
        ['item', 'item_name', 'style', 'gender', 'season', 'tpo']
    ]
    return result

sample_user = rate_df['user'].iloc[0]
print(f"유저 {sample_user} 추천 Top 10:")
print(recommend_items(sample_user))

유저 27 추천 Top 10:
        item              item_name style gender              season  \
10281  10281   W_00076_50_ivy_M.jpg   ivy      M         spring fall   
11766  11766   W_01567_50_ivy_M.jpg   ivy      M         spring fall   
14395  14395   W_04205_50_ivy_M.jpg   ivy      M         spring fall   
16424  16424  W_06242_60_mods_M.jpg  mods      M  spring fall summer   
17011  17011  W_06830_60_mods_M.jpg  mods      M  spring fall winter   
17410  17410   W_07236_50_ivy_M.jpg   ivy      M         spring fall   
19418  19418   W_09250_50_ivy_M.jpg   ivy      M         spring fall   
20940  20940   W_10782_50_ivy_M.jpg   ivy      M         spring fall   
22956  22956   W_12813_50_ivy_M.jpg   ivy      M  spring fall winter   
27540  27540   W_17423_50_ivy_M.jpg   ivy      M  spring fall winter   

                                                     tpo  
10281                                         attendance  
11766                                   daily date event  
14395        

In [8]:
item_stats = rate_df.groupby('item').agg(
    avg_rate=('rate', 'mean'),
    count=('rate', 'count')
).reset_index()

item_stats['score'] = item_stats['avg_rate'] * np.log1p(item_stats['count'])

top10 = item_stats.sort_values('score', ascending=False).head(10)
top10 = top10.merge(
    item_df[['item', 'item_name', 'style', 'gender', 'season']], on='item'
)

print(top10[['item_name', 'style', 'gender', 'season', 'score']].to_string())

                         item_name           style gender                     season     score
0             W_26236_50_ivy_M.jpg             ivy      M         spring fall winter  5.941262
1            W_06149_60_mods_M.jpg            mods      M         spring fall winter  5.792730
2             W_06258_50_ivy_M.jpg             ivy      M         spring fall summer  5.792730
3             W_15370_50_ivy_M.jpg             ivy      M  spring fall summer winter  5.792730
4             W_09874_50_ivy_M.jpg             ivy      M         spring fall winter  5.792730
5  W_05806_10_sportivecasual_W.jpg  sportivecasual      W         spring fall summer  5.675571
6  W_09037_10_sportivecasual_W.jpg  sportivecasual      W         spring fall winter  5.675571
7             W_04645_50_ivy_M.jpg             ivy      M                     summer  5.644198
8             W_24016_50_ivy_M.jpg             ivy      M         spring fall winter  5.644198
9             W_07109_50_ivy_M.jpg             ivy

In [1]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 46.2 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import implicit
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt

# 데이터 로드
@st.cache_data
def load_data():
    rate_df = pd.read_csv('rate_data.csv', encoding='utf-8-sig')
    item_df = pd.read_csv('item_data.csv', encoding='utf-8-sig')
    return rate_df, item_df

# 모델 학습
@st.cache_resource
def train_model(rate_df):
    user_ids = rate_df['user'].astype('category').cat.codes
    item_ids = rate_df['item'].astype('category').cat.codes
    sparse_matrix = csr_matrix((rate_df['rate'], (user_ids, item_ids)))
    model = implicit.als.AlternatingLeastSquares(
        factors=50, iterations=30, use_gpu=False
    )
    model.fit(sparse_matrix)
    return model, sparse_matrix

rate_df, item_df = load_data()
model, sparse_matrix = train_model(rate_df)

# 페이지 설정
st.set_page_config(page_title="패션 사입 추천 시스템", page_icon="👗", layout="wide")
st.title("👗 패션 사입 추천 시스템")
st.markdown("**트렌드 데이터 기반으로 사입할 아이템을 추천해드립니다**")

# 사이드바
st.sidebar.header("🔍 조건 입력")
gender = st.sidebar.selectbox("성별", ["전체", "M (남성)", "W (여성)"])
season = st.sidebar.selectbox("시즌", ["전체", "spring", "summer", "fall", "winter"])
styles = ["전체"] + sorted(item_df['style'].unique().tolist())
style = st.sidebar.selectbox("스타일", styles)
top_n = st.sidebar.slider("추천 개수", 5, 20, 10)

# 필터링
filtered = item_df.copy()
if gender != "전체":
    filtered = filtered[filtered['gender'] == gender.split()[0]]
if season != "전체":
    filtered = filtered[filtered['season'].str.contains(season, na=False)]
if style != "전체":
    filtered = filtered[filtered['style'] == style]

# 트렌드 점수 계산
item_stats = rate_df.groupby('item').agg(
    avg_rate=('rate', 'mean'),
    count=('rate', 'count')
).reset_index()
item_stats['score'] = item_stats['avg_rate'] * np.log1p(item_stats['count'])

result = filtered.merge(item_stats, on='item').sort_values('score', ascending=False).head(top_n)

# 메인 화면
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("📊 스타일별 트렌드")
    style_score = rate_df.merge(
        item_df[['item', 'style']], on='item'
    ).groupby('style')['rate'].mean().sort_values(ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(style_score.index, style_score.values, color='tomato')
    ax.set_title('Top 10 트렌드 스타일')
    ax.set_xlabel('스타일')
    ax.set_ylabel('평균 선호도')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    st.pyplot(fig)

with col2:
    st.subheader("🛍️ 추천 사입 아이템")
    if result.empty:
        st.warning("조건에 맞는 아이템이 없어요. 조건을 바꿔보세요!")
    else:
        display = result[['item_name', 'style', 'gender', 'season', 'score']].copy()
        display.columns = ['아이템명', '스타일', '성별', '시즌', '추천점수']
        display['추천점수'] = display['추천점수'].round(2)
        display = display.reset_index(drop=True)
        display.index += 1
        st.dataframe(display, use_container_width=True)

st.markdown("---")
st.caption("AI Hub 연도별 패션 선호도 데이터 기반 | 사회과학기반 디자인씽킹 BM")

Writing app.py


In [10]:
from pyngrok import ngrok
import subprocess, time
import getpass

# 코드를 실행하면 아래에 입력창이 뜰 거예요. 거기에 토큰을 붙여넣으시면 됩니다!
ngrok_token = getpass.getpass("ngrok 토큰을 입력하세요: ")
ngrok.set_auth_token(ngrok_token)

proc = subprocess.Popen(['streamlit', 'run', 'app.py',
                         '--server.port', '8501'])
time.sleep(3)
tunnel = ngrok.connect(8501)
print("🚀 웹앱 주소:", tunnel.public_url)

ngrok 토큰을 입력하세요: ··········
🚀 웹앱 주소: https://underfoot-distill-outplayed.ngrok-free.dev
